In [3]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [49]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
from flatten_json import flatten

In [50]:
# Automated date and time
start1 = dt.datetime(2019,5,11)
start = start.replace(hour=0, minute=0, second=0, microsecond=0)
start = start - dt.timedelta(hours = 5, minutes = 30)
end1 = dt.datetime(2019,5,12)
end = end.replace(hour=0, minute=0, second=0, microsecond=0)
end = end - dt.timedelta(hours = 5, minutes = 30)
print(start,end,end-start)

2019-05-09 18:30:00 2019-05-10 18:30:00 1 day, 0:00:00


In [54]:
table_str1 = ("`hitwicketsuperstars.analytics_190927423.new_user_reference`")
print(table_str1)

`hitwicketsuperstars.analytics_190927423.new_user_reference`


In [62]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (f"""SELECT
        *
        FROM
       
            AND _TABLE_SUFFIX BETWEEN '0509'AND '0512'"""
        )
df = client.query(query).to_dataframe()
df.head()

BadRequest: 400 Syntax error: Unexpected keyword AND at [6:13]

In [ ]:

client = bigquery.Client(project='hitwicketsuperstars')
query = (
    f"""SELECT
             event_name,
             user_id,
             event_timestamp,
             device.mobile_os_hardware_model
     FROM `hitwicketsuperstars.analytics_190927423.events_2019*`
             AND _TABLE_SUFFIX BETWEEN '0509'AND '0512'"""
)
df = client.query(query).to_dataframe()
df.head()

In [18]:
print(len(events))
events.head()

9995


,__v,_id,action,app_platform,app_version,created_at,device_id,element,page,updated_at
0,0,5cd5c341eeef9931e8b59d03,SEEN,UNITY_Android,11,2019-05-10 18:30:25.402,a35785792ad73508bba100ca8f63cd85,SPLASH,FIRSTTIME,2019-05-10 18:30:25.402
1,0,5cd5c34beeef9931e8b59d06,SEEN,UNITY_Android,11,2019-05-10 18:30:35.273,36e950740f77605a1f587f9307d814e4,SPLASH,FIRSTTIME,2019-05-10 18:30:35.273
2,0,5cd5c386eeef9931e8b59d27,SEEN,UNITY_Android,11,2019-05-10 18:31:34.025,cb19ea6c529ee2e2092ef775035da39a,SPLASH,FIRSTTIME,2019-05-10 18:31:34.025
3,0,5cd5c38beeef9931e8b59d2f,SEEN,UNITY_Android,11,2019-05-10 18:31:39.895,b45e1fe7c9cf8d7201514164fc8dc4a4,SPLASH,FIRSTTIME,2019-05-10 18:31:39.895
4,0,5cd5c3b2eeef9931e8b59d47,SEEN,UNITY_Android,11,2019-05-10 18:32:18.306,9b54c1f5ea719d3602da532a42e74054,SPLASH,FIRSTTIME,2019-05-10 18:32:18.306


In [21]:
events = events.sort_values(by = ["device_id","created_at"])
events = events.drop_duplicates("device_id")
opened_apps_first_time = len(events)
len(events)

8598

In [26]:
devices = events["device_id"].tolist()

In [22]:
table_str1 = ("`hitwicketsuperstars.analytics_190927423.events_" + 
             str(start.date()) + "`")
table_str1 = re.sub('-','',table_str1)
table_str2 = ("`hitwicketsuperstars.analytics_190927423.events_" + 
             str(end.date()) + "`")
table_str2 = re.sub('-','',table_str2)
print(table_str1,table_str2)

`hitwicketsuperstars.analytics_190927423.events_20190510` `hitwicketsuperstars.analytics_190927423.events_20190511`


In [23]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (
    f"""SELECT
  user_id,
  event_name,
  event_timestamp,
  params.key,
  params.value.string_value,
  device
FROM 
  (SELECT
      event_name,
      user_id,
      event_timestamp,
      event_params,
      device.mobile_os_hardware_model as device
  FROM
      {table_str1}
  UNION ALL
  SELECT
      event_name,
      user_id,
      event_timestamp,
      event_params,
      device.mobile_os_hardware_model as device
  FROM
      {table_str2}
  ),
  UNNEST(event_params) AS params
  WHERE params.key = 'action'"""
)
df = client.query(query).to_dataframe()
df.head()

,user_id,event_name,event_timestamp,key,string_value,device
0,4294584d2aeea8dcba960656f9c10cd5,ftue,1557502568561000,action,hand_pointer_playing_eleven3_clicked,T1
1,4294584d2aeea8dcba960656f9c10cd5,ftue,1557502593861003,action,hand_pointer_rivals_vs_match31seen,T1
2,4294584d2aeea8dcba960656f9c10cd5,ftue,1557502595532004,action,hand_pointer_rivals_vs_match31clicked,T1
3,4294584d2aeea8dcba960656f9c10cd5,quick_match,1557502625756006,action,toss_started,T1
4,4294584d2aeea8dcba960656f9c10cd5,quick_match,1557502653871000,action,toss_ended,T1


In [24]:
len(df)

1881800

In [27]:
map1 = pd.read_csv("/home/aurora/analytics/scratchpad/Maps/mod_map1.csv")
map1.head()

,Label,Action,Order
0,Opened App,opened_app,0
1,New User Hand Pointer Clicked,new_user_hand_pointer_click,1
2,First Welcome Natasha Seen,natasha_welcome1_seen,2
3,First Welcome Natasha Clicked,natasha_welcome1_clicked,3
4,Second Welcome Natasha Seen,natasha_welcome2_seen,4


In [30]:
user_funnel = df[df["user_id"].isin(devices)]
user_funnel = user_funnel.sort_values(by = ["user_id","event_timestamp"])
user_funnel = user_funnel[user_funnel["string_value"].isin(map1["Action"])]
user_funnel.drop_duplicates(subset = ['user_id','string_value'], inplace = True)
len(user_funnel)

587101

In [31]:
user_funnel = user_funnel.groupby('string_value').agg({'user_id' : 'count'})

In [32]:
user_funnel.head()

,user_id
string_value,
crete_team_next_clicked,2730
hand_pointer_dash_scout_01clicked,7692
hand_pointer_dash_scout_01seen,7725
hand_pointer_dash_scout_11clicked,3221
hand_pointer_dash_scout_11seen,3451


In [33]:
user_funnel = pd.merge(user_funnel, map1, left_index = True, right_on = 'Action')
user_funnel = user_funnel.sort_values(by = 'Order')
user_funnel.head()

,user_id,Label,Action,Order
1,8537,New User Hand Pointer Clicked,new_user_hand_pointer_click,1
2,8405,First Welcome Natasha Seen,natasha_welcome1_seen,2
3,8104,First Welcome Natasha Clicked,natasha_welcome1_clicked,3
4,8103,Second Welcome Natasha Seen,natasha_welcome2_seen,4
5,7991,Second Welcome Natasha Clicked,natasha_welcome2_clicked,5


In [34]:
opened_app_df = pd.DataFrame({'user_id' : [opened_apps_first_time], "Label" : 'Opened App',
                            'Action' : 'Opened App', 'Order' : 0})
user_funnel = opened_app_df.append(user_funnel)

In [35]:
user_funnel.head()

,user_id,Label,Action,Order
0,8598,Opened App,Opened App,0
1,8537,New User Hand Pointer Clicked,new_user_hand_pointer_click,1
2,8405,First Welcome Natasha Seen,natasha_welcome1_seen,2
3,8104,First Welcome Natasha Clicked,natasha_welcome1_clicked,3
4,8103,Second Welcome Natasha Seen,natasha_welcome2_seen,4


In [44]:
final = user_funnel
final.columns=['no of users','label','action','order']

In [45]:
final.head()

,no of users,label,action,order
0,8598,Opened App,Opened App,0
1,8537,New User Hand Pointer Clicked,new_user_hand_pointer_click,1
2,8405,First Welcome Natasha Seen,natasha_welcome1_seen,2
3,8104,First Welcome Natasha Clicked,natasha_welcome1_clicked,3
4,8103,Second Welcome Natasha Seen,natasha_welcome2_seen,4
